# AdaBoost

"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base. One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering - Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information. The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being. However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.

In [ ]:
# importing libs:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict

In [ ]:
df = pd.read_csv("../../../../../data/Travel.csv")
df.head()

## Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns
non_numeric_cols = df.select_dtypes(exclude='number').columns
numeric_cols

In [ ]:
non_numeric_cols

In [ ]:
for col in non_numeric_cols:
   print(df[col].value_counts(),"\n")

In [ ]:
for col in non_numeric_cols:
   print(col ,": ", df[col].unique())

Gender and Marital Status have non uniform entries, to correct them:

In [ ]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['Gender'].unique()

In [ ]:
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')
df['MaritalStatus'].unique()

In [ ]:
# reducing unnecessary columns
df['Visited'] = df['NumberOfChildrenVisiting']+df["NumberOfPersonVisiting"]
df.head()


In [ ]:
df = df.drop(columns=['NumberOfChildrenVisiting', 'NumberOfPersonVisiting'])
df.head()

In [ ]:
# missing values


In [ ]:
df.isnull().sum()[df.isnull().sum() >= 1]


In [ ]:
features_with_na = [
    features for features in df.columns if df[features].isnull().sum()>=1
]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100,5),"% missing values")

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent[null_percent >= 1]
null_percent.round(5)

In [ ]:
null_summary = (
    df.isnull()
      .mean()
      .mul(100)
      .round(5)
      .to_frame(name="percent_missing")
)

null_summary = null_summary[null_summary["percent_missing"] >= 1]


In [ ]:
null_summary.sort_values("percent_missing", ascending=False)


In [ ]:
df[features_with_na].select_dtypes(exclude='O').describe()

## Split and basic analysis

In [ ]:
# Numerical features:
num_features = df.select_dtypes(include='number').columns.tolist()
print(f'Number of Numerical Features: {len(num_features)}')

# categorical features:
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Number of Categorical Features: {len(cat_features)}')

# Discrete features:
discrete_features = [
    f for f in num_features if df[f].nunique() <= 25
]
print(f'Number of Discrete Features: {len(discrete_features)}')

# Continuous features:
con_features = [
    f for f in num_features if f not in discrete_features
]
print(f'Number of Continuous Features: {len(con_features)}')

In [ ]:
X = df.drop(columns="ProdTaken")
y = df['ProdTaken']

In [ ]:
X.head()

In [ ]:
y.value_counts()

In [ ]:
plt.scatter(range(len(y)), y)
plt.xlabel('Index')
plt.ylabel('y')
plt.title('Scatter plot of y')
plt.show()

In [ ]:
# Count of 0s and 1s
counts = y.value_counts()

plt.bar(counts.index, counts.values, color=['skyblue', 'salmon'])
plt.xticks([0, 1], ['No', 'Yes'])  # optional: rename 0/1 to No/Yes
plt.ylabel('Count')
plt.title('Distribution of binary target y')
plt.show()


In [ ]:
num_features

In [ ]:
for cols in num_features:
    sns.countplot(x=cols, hue=y, data=X.join(y))
    plt.ylabel('Count')
    plt.title(f'Distribution of y with respect to {cols}')
    plt.show()


In [ ]:
numeric_X = X.select_dtypes(include='number')

# Pairplot of numeric features
sns.pairplot(numeric_X)

In [ ]:
y_named = y.copy()
y_named.name = 'y'  # set the column name

sns.pairplot(numeric_X.join(y_named), hue='y')


In [ ]:
plt.figure(figsize=(9, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")

In [ ]:
plt.figure(figsize=(9, 8))
sns.heatmap(X.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")

## Pipelines and Training

In [ ]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
# setting up pipelines:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

discrete_num_features = ['Visited']
continuous_num_features = ['Age', 'DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']

# Discrete pipeline (mode)
discrete_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

# Continuous pipeline (median + scale)
continuous_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # fill missing with mode
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))     # convert to 0/1 vectors
])

preprocessor = ColumnTransformer([
    ('cont', continuous_pipeline, continuous_num_features),
    ('disc', discrete_pipeline, discrete_num_features),
    ('cat', cat_pipeline, cat_features)
])

In [ ]:
models = {
    "AdaBoost": AdaBoostClassifier(),
}
preprocessor


In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
df.columns

In [ ]:
pipelines['AdaBoost']

In [ ]:
def get_classification_metrics(y_true, y_pred, y_proba=None, model_name=None, verbose=True):
    """
    Calculate standard classification metrics and optionally print them.

    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_proba : array-like, optional
        Predicted probabilities for positive class (for ROC AUC)
    model_name : str, optional
        Name of the model for printing
    verbose : bool
        Whether to print the metrics

    Returns:
    --------
    metrics : dict
        Dictionary containing accuracy, precision, recall, f1, roc_auc, confusion_matrix
    """
    metrics = {}
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred)
    metrics['recall'] = recall_score(y_true, y_pred)
    metrics['f1'] = f1_score(y_true, y_pred)
    if y_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    else:
        metrics['roc_auc'] = None

    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred)
    if verbose:
        if model_name:
            print(f"--- {model_name} ---")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"F1-score : {metrics['f1']:.4f}")
        print(f"ROC AUC  : {metrics['roc_auc']}")
        print("Confusion Matrix:")
        print(metrics['confusion_matrix'])
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred))
        print("-"*40)
    return metrics


In [ ]:
X = df.drop('ProdTaken', axis=1)
y = df['ProdTaken']

X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25, random_state=67
    )
for name, cols in [('continuous', continuous_num_features),
                   ('discrete', discrete_num_features),
                   ('categorical', cat_features)]:
    print(f"{name} columns:", cols)

# now transformation:
X_train_transformed = preprocessor.fit_transform(X_train)
X_train_transformed=pd.DataFrame(X_train_transformed)
pd.DataFrame(X_train_transformed)

In [ ]:
X_test_transformed = pd.DataFrame(preprocessor.transform(X_test))
X_test_transformed

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


- **Accuracy:** The proportion of total correct predictions (both classes). On the test set, 88.5% of samples are correctly classified.
- **Precision:** Of all predicted positives, how many were actually positive. Here, ~79.5% of predicted '1's are correct.
- **Recall (Sensitivity):** Of all actual positives, how many were correctly predicted. Only ~45.8% of positive cases were identified → many false negatives.
- **F1-score:** Harmonic mean of precision and recall; balances both. Here, 0.5808 indicates moderate performance on the positive class.
- **ROC AUC:** Probability that a randomly chosen positive ranks higher than a randomly chosen negative. 0.8923 shows good ranking ability despite low recall.
- **Confusion Matrix:** Displays counts of true negatives, false positives, false negatives, and true positives. Shows the model missed 115 positives in the test set.


**Bad model lets try the whole set of it now:**

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(),
    "AdaBoost": AdaBoostClassifier(),
}

In [ ]:
pipelines = {name: Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
]) for name, model in models.items()}
pipelines

In [ ]:
results = {}

for name, pipe in pipelines.items():
    print(f"Training {name}...")

    # Train the model
    pipe.fit(X_train, y_train)

    # TRAIN metrics
    y_train_pred = pipe.predict(X_train)
    try:
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None
    train_metrics = get_classification_metrics(y_train, y_train_pred, y_train_proba,verbose=True)

    # TEST metrics
    y_test_pred = pipe.predict(X_test)
    try:
        y_test_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None
    test_metrics = get_classification_metrics(y_test, y_test_pred, y_test_proba,verbose=True)

    results[name] = {'train': train_metrics, 'test': test_metrics}


In [ ]:
dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "saga"],
    "model__class_weight": [None, "balanced"]
}
rf_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__max_depth": [None, 5, 8, 10, 15],
    "model__max_features": ["sqrt", 5, 7, 8],
    "model__min_samples_split": [2, 8, 15, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True, False]
}
svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50, 100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3)
    ]
}
randomcv_models = [
    ("Decision Tree",
     DecisionTreeClassifier(),
     dt_params),

    ("Logistic Regression",
     LogisticRegression(max_iter=1000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(random_state=42),
     rf_params),

    # ("SVM",
    #  SVC(probability=True),
    #  svm_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]

In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=30,
        scoring='f1',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=150,
        scoring='f1',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


## Lets make it more better:

In [ ]:
dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "saga"],
    "model__class_weight": [None, "balanced"]
}
rf_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__max_depth": [None, 5, 8, 10, 15],
    "model__max_features": ["sqrt", 5, 7, 8],
    "model__min_samples_split": [2, 8, 15, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True, False]
}
svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50, 100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4)
    ]
}
randomcv_models = [("Decision Tree",
     DecisionTreeClassifier(class_weight="balanced"),
     dt_params),

    ("Logistic Regression",
     LogisticRegression(max_iter=5000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(class_weight="balanced_subsample"),
     rf_params),

    # ("SVM",
    #  SVC(probability=True),
    #  svm_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]

In [ ]:
dt_params = {
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__criterion": ["gini", "entropy"]
}
lr_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "saga"],
    "model__class_weight": [None, "balanced"]
}
rf_params = {
    "model__n_estimators": [100, 200, 500, 1000],
    "model__max_depth": [None, 5, 8, 10, 15],
    "model__max_features": ["sqrt", 5, 7, 8],
    "model__min_samples_split": [2, 8, 15, 20],
    "model__min_samples_leaf": [1, 2, 4],
    "model__bootstrap": [True, False]
}
svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf", "poly"],
    "model__gamma": ["scale", "auto"],
    "model__class_weight": [None, "balanced"]
}
ada_params = {
    "model__n_estimators": [50, 100, 200, 500],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4)
    ]
}
randomcv_models = [("Decision Tree",
     DecisionTreeClassifier(class_weight="balanced"),
     dt_params),

    ("Logistic Regression",
     LogisticRegression(max_iter=5000),
     lr_params),

    ("Random Forest",
     RandomForestClassifier(class_weight="balanced_subsample"),
     rf_params),

    # ("SVM",
    #  SVC(probability=True),
    #  svm_params),

    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]

In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=100,
        scoring='f1_macro',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=100,
        scoring='f1_macro',
        cv=5,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


## Looking more into top 2 models (RF and AdaBoost)

In [ ]:
rf_params = {
    "model__n_estimators": [400, 600, 800, 1000,],
    "model__max_depth": [None, 10, 30],
    "model__max_features": ["sqrt", "log2", 5, 6, 7,8],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 3, 4, 5],
    "model__bootstrap": [True, False],
    "model__criterion": ["gini", "entropy", "log_loss"]
}
ada_params = {
    "model__n_estimators": [50,70,100,150,200, 300, 400],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0, 1.2],
    "model__algorithm":['SAMME'],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4),
    ]
}


random_search = RandomizedSearchCV(
    pipe,
    param_distributions=params,
    n_iter=200,        # larger iteration budget
    scoring='f1_macro',
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)
randomcv_models = [
    ("Random Forest",
     RandomForestClassifier(class_weight="balanced_subsample"),
     rf_params),
    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]


In [ ]:
best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=75,
        scoring='f1_macro',
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )


    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
# rf_params = {
#     "model__n_estimators": [400, 600, 800, 1000,],
#     "model__max_depth": [None, 10, 30],
#     "model__max_features": ["sqrt", "log2", 5, 6, 7,8],
#     "model__min_samples_split": [2, 4, 6, 8],
#     "model__min_samples_leaf": [1, 2, 3, 4, 5],
#     "model__bootstrap": [True, False],
#     "model__criterion": ["gini", "entropy", "log_loss"]
# }
ada_params = {
    "model__n_estimators": [50,70,100,150,200, 300, 400],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0, 1.2],
    "model__algorithm":['SAMME'],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4),
    ]
}


# random_search = RandomizedSearchCV(
#     pipe,
#     param_distributions=params,
#     n_iter=200,        # larger iteration budget
#     scoring='f1_macro',
#     cv=5,
#     verbose=1,
#     n_jobs=-1,
#     random_state=42
# )
randomcv_models = [
    # ("Random Forest",
    #  RandomForestClassifier(class_weight="balanced_subsample"),
    #  rf_params),
    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]

# best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=75,
        scoring='f1_macro',
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )


    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
best_models

In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
# Example: AdaBoost threshold tuning
pipe = best_models["AdaBoost"]

y_train_proba_cv = cross_val_predict(
    pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
f1_scores = []

for t in thresholds:
    y_pred = (y_train_proba_cv >= t).astype(int)
    f1_scores.append(f1_score(y_train, y_pred))

best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"Best threshold: {best_threshold:.2f}")
print(f"CV F1-score  : {best_f1:.4f}")

# Apply threshold to test set
y_test_proba = pipe.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(int)

get_classification_metrics(y_test, y_test_pred, y_test_proba, verbose=True)


In [ ]:
# Define models
models_to_eval = {
    "Random Forest": best_models["Random Forest"],
    "AdaBoost": best_models["AdaBoost"]
}

results_thresholds = {}

plt.figure(figsize=(8, 6))

for name, pipe in models_to_eval.items():
    # CV probabilities for threshold tuning
    y_train_proba_cv = cross_val_predict(
        pipe, X_train, y_train, cv=5, method="predict_proba", n_jobs=-1
    )[:, 1]

    # Find F1-optimal threshold
    thresholds = np.linspace(0.05, 0.95, 91)
    f1_scores = [f1_score(y_train, (y_train_proba_cv >= t).astype(int)) for t in thresholds]
    best_threshold = thresholds[np.argmax(f1_scores)]
    best_f1 = max(f1_scores)

    # Predict on test set using best threshold
    y_test_proba = pipe.predict_proba(X_test)[:, 1]
    y_test_pred_f1 = (y_test_proba >= best_threshold).astype(int)

    # ROC curve & AUC
    fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_proba)
    auc_score = roc_auc_score(y_test, y_test_proba)

    # Youden's J for ROC-based threshold
    j_scores = tpr - fpr
    roc_threshold = roc_thresholds[np.argmax(j_scores)]

    # Plot ROC
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

    # Save results
    results_thresholds[name] = {
        "best_f1_threshold": best_threshold,
        "best_f1_score": best_f1,
        "roc_threshold": roc_threshold,
        "auc": auc_score,
        "y_test_pred_f1": y_test_pred_f1
    }

# Random baseline
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Print thresholds
for name, info in results_thresholds.items():
    print(f"\n{name}:")
    print(f" - Best F1 threshold: {info['best_f1_threshold']:.3f} (CV F1 = {info['best_f1_score']:.4f})")
    print(f" - ROC-based threshold: {info['roc_threshold']:.3f} (AUC = {info['auc']:.3f})")


In [ ]:
# Example for Random Forest with F1-optimal threshold:
best_rf_threshold = 0.430

y_test_proba_rf = best_models["Random Forest"].predict_proba(X_test)[:, 1]
y_test_pred_rf = (y_test_proba_rf >= best_rf_threshold).astype(int)

# Similarly for AdaBoost:
best_ada_threshold = 0.480

y_test_proba_ada = best_models["AdaBoost"].predict_proba(X_test)[:, 1]
y_test_pred_ada = (y_test_proba_ada >= best_ada_threshold).astype(int)


## GPT sensei:

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, roc_auc_score

class ThresholdClassifier:
    def __init__(self, pipeline, threshold=0.5):
        self.pipeline = pipeline
        self.threshold = threshold
    
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)
    
    def predict(self, X):
        proba = self.pipeline.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return f1_score(y, y_pred)

def print_metrics(y_true, y_pred, y_proba=None):
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")
    if y_proba is not None:
        print(f"ROC AUC  : {roc_auc_score(y_true, y_proba):.4f}")

# Wrap pipelines with thresholds
rf_with_threshold = ThresholdClassifier(best_models["Random Forest"], threshold=0.43)
ada_with_threshold = ThresholdClassifier(best_models["AdaBoost"], threshold=0.48)

# Optional: retrain on full train set if not trained already
rf_with_threshold.fit(X_train, y_train)
ada_with_threshold.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_with_threshold.predict(X_test)
y_pred_ada = ada_with_threshold.predict(X_test)

# Get probabilities for ROC-AUC
y_proba_rf = rf_with_threshold.predict_proba(X_test)[:, 1]
y_proba_ada = ada_with_threshold.predict_proba(X_test)[:, 1]

print("=== Random Forest Metrics with Threshold 0.43 ===")
print_metrics(y_test, y_pred_rf, y_proba_rf)

print("\n=== AdaBoost Metrics with Threshold 0.48 ===")
print_metrics(y_test, y_pred_ada, y_proba_ada)


## elaboration of metrics:
1. Confusion Matrix

Definition: A table showing true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN) for predicted vs actual classes.

Interpretation: For Random Forest, 150 correctly predicted positives (TP) and 943 correctly predicted negatives (TN) show strong class separation, while 67 false positives and 62 false negatives indicate the remaining errors.

2. Accuracy

Definition: Fraction of total correct predictions over all predictions: (TP + TN) / total.

Interpretation: Random Forest achieves 89.4% accuracy, meaning overall it classifies ~9 out of 10 samples correctly; AdaBoost is 80.1%, slightly less accurate overall but still decent.

3. Precision

Definition: Fraction of correctly predicted positives out of all predicted positives: TP / (TP + FP).

Interpretation: RF precision for class 1 is 0.69 → when it predicts “1”, it’s correct 69% of the time; AdaBoost only 0.45 → more false positives for positives.

4. Recall

Definition: Fraction of correctly predicted positives out of all actual positives: TP / (TP + FN).

Interpretation: RF recall for class 1 is 0.71 → it catches 71% of actual positive cases; AdaBoost recall 0.65 → misses more positives than RF.

5. F1-score

Definition: Harmonic mean of precision and recall: 2 * (precision * recall) / (precision + recall).

Interpretation: RF F1-score 0.70 → balances precision & recall well for positives; AdaBoost 0.53 → less balanced, favors recall slightly over precision.

6. ROC AUC

Definition: Area under the Receiver Operating Characteristic curve; measures ability to rank positive samples higher than negatives.

Interpretation: RF AUC 0.92 → very good discrimination between classes; AdaBoost 0.83 → moderately good, but less confident in ranking positives vs negatives.

7. Macro vs Weighted Avg (from classification report)

Macro avg: Simple average of metrics across classes, ignoring class imbalance.

Weighted avg: Average weighted by class support → more realistic for imbalanced data.

Interpretation: Macro F1 for RF is 0.82 → balanced performance; weighted F1 0.89 → performance heavily influenced by majority class (negatives).

| Metric                                | Definition                                    | Random Forest (Threshold 0.43) | Meaning                                                              | AdaBoost (Threshold 0.48) | Meaning                                                          |
| ------------------------------------- | --------------------------------------------- | ------------------------------ | -------------------------------------------------------------------- | ------------------------- | ---------------------------------------------------------------- |
| **Accuracy**                          | Correct predictions / total samples           | 0.894                          | ~89% of all samples classified correctly                             | 0.801                     | ~80% correct overall; slightly worse than RF                     |
| **Precision (class 1)**               | TP / (TP + FP)                                | 0.69                           | When predicting positive, correct 69% of the time                    | 0.45                      | More false positives; only 45% of positive predictions correct   |
| **Recall (class 1)**                  | TP / (TP + FN)                                | 0.71                           | Captures 71% of actual positives                                     | 0.65                      | Captures 65% of actual positives; misses more than RF            |
| **F1-score (class 1)**                | Harmonic mean of precision & recall           | 0.70                           | Balanced performance between precision & recall                      | 0.53                      | Less balanced; favors recall slightly                            |
| **ROC AUC**                           | Probability ranking of positives vs negatives | 0.920                          | Excellent separation of classes                                      | 0.833                     | Good separation but less confident than RF                       |
| **Macro F1**                          | Avg F1 ignoring class imbalance               | 0.82                           | Strong performance across both classes                               | 0.70                      | Performance weaker across both classes                           |
| **Weighted F1**                       | Avg F1 weighted by support                    | 0.89                           | Weighted by majority class; overall strong                           | 0.81                      | Overall performance influenced by majority class; weaker than RF |
| **Confusion Matrix (TP, TN, FP, FN)** | Counts of prediction vs actual                | [[943, 67],[62, 150]]          | Most negatives & positives correctly predicted; errors: 67 FP, 62 FN | [[841, 169],[74, 138]]    | More errors for both classes; positives underpredicted           |


# SUMMARY:


---

## **1. Data Preprocessing**

Before any modeling, you set up a **pipeline for preprocessing**, because machine learning models require **numerical input** and consistent handling of missing values:

* **Continuous numeric features** (`['Age', 'DurationOfPitch', 'NumberOfTrips', 'MonthlyIncome']`):

  * Imputed with **median** to handle missing values.
  * Scaled using **StandardScaler** so features are centered and scaled.

* **Discrete numeric feature** (`['Visited']`):

  * Imputed with **mode** (most frequent value).
  * Scaled using **StandardScaler**.

* **Categorical features** (`['TypeofContact', 'Occupation', 'Gender', 'ProductPitched', 'MaritalStatus', 'Designation']`):

  * Imputed with **mode**.
  * Encoded using **OneHotEncoder** with `drop='first'` to avoid multicollinearity.

All of this was combined in a **`ColumnTransformer`**, so we can preprocess different types of features automatically.

---

## **2. Setting up Models and Hyperparameter Grids**

You wanted to compare **Random Forest (RF)** and **AdaBoost**, so we defined **exhaustive parameter grids** for hyperparameter search:

* **Random Forest (`rf_params`)**:

  * `n_estimators`: number of trees.
  * `max_depth`: max depth of trees.
  * `max_features`: features considered for each split.
  * `min_samples_split` / `min_samples_leaf`: control overfitting.
  * `bootstrap`: sampling strategy.
  * `criterion`: split criterion (`gini`, `entropy`, `log_loss`).

* **AdaBoost (`ada_params`)**:

  * `n_estimators`: number of boosting rounds.
  * `learning_rate`: weight of each weak learner.
  * `algorithm`: boosting type (`SAMME`).
  * `estimator`: the weak learner, in this case decision trees of depths 1–4.

**Why we do this:**

Hyperparameter tuning finds the **best model configuration** that maximizes a chosen metric (here `f1_macro`) **without overfitting**.

---

## **3. Hyperparameter Tuning with RandomizedSearchCV**

We used **`RandomizedSearchCV`**:

* Wrap each model with a **pipeline** including the preprocessor.
* Run cross-validation (CV) over a random selection of parameter combinations.
* Evaluate using `f1_macro` to handle class imbalance.
* Choose the **best parameters** automatically.

```python
best_models[name] = random_search.best_estimator_
```

**Outcome:**

We got **best models** for Random Forest and AdaBoost, trained with hyperparameters that gave the highest CV F1-score.

---

## **4. Generating Probabilities and Finding Optimal Threshold**

Most classifiers output **probabilities**:

```python
proba_rf = best_models["Random Forest"].predict_proba(X_train)[:, 1]
```

Default threshold for classification is **0.5**, but for imbalanced classes, this **might not maximize F1 or other metrics**.

So, we searched for the **optimal threshold**:

1. Evaluate F1-score for thresholds from 0.05 to 0.95.
2. Choose threshold that **maximizes F1-score** (`best_threshold`).
3. Also calculated **ROC-based threshold** (Youden’s J statistic: `tpr - fpr`) to optimize **sensitivity vs specificity**.

**Example results:**

| Model         | Best F1 Threshold | ROC-based Threshold | AUC   |
| ------------- | ----------------- | ------------------- | ----- |
| Random Forest | 0.43              | 0.305               | 0.920 |
| AdaBoost      | 0.48              | 0.475               | 0.833 |

**Why:** Threshold tuning is **post-training**—we don’t retrain the model; we just adjust the cutoff for converting probabilities → classes.

---

## **5. Plugging Thresholds Back into Pipelines**

We created a **`ThresholdClassifier`** class to wrap your pipeline:

```python
class ThresholdClassifier:
    def __init__(self, pipeline, threshold=0.5):
        self.pipeline = pipeline
        self.threshold = threshold

    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

    def predict(self, X):
        proba = self.pipeline.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)
```

* Keeps the **pipeline and preprocessing intact**.
* Applies **custom threshold** automatically during `.predict()`.
* F1 or accuracy can be evaluated directly through `.score()` if you want.

Then we wrapped our best models:

```python
rf_with_threshold = ThresholdClassifier(best_models["Random Forest"], threshold=0.43)
ada_with_threshold = ThresholdClassifier(best_models["AdaBoost"], threshold=0.48)
```

Now every call to `.predict(X_test)` uses the **optimized threshold** automatically.

---

## **6. Final Predictions and Metrics**

* Predict probabilities → apply threshold → get final predictions.
* Evaluate **accuracy, F1, ROC-AUC**, confusion matrix, and classification report.

**Example output:**

**Random Forest (Threshold 0.43):**

* Accuracy: 0.894
* F1: 0.699
* ROC AUC: 0.920

**AdaBoost (Threshold 0.48):**

* Accuracy: 0.801
* F1: 0.532
* ROC AUC: 0.833

> Notice: The threshold adjustment improved F1-score for the **minority class** without retraining the models.

---

## ✅ **7. Summary of Flow**

1. **Preprocessing pipeline** → handle missing data, scale continuous and discrete features, encode categoricals.
2. **Define models** → Random Forest and AdaBoost, with hyperparameter grids.
3. **Hyperparameter tuning** → `RandomizedSearchCV` to find best model configurations.
4. **Train models** on the full training set using best parameters.
5. **Generate probabilities** on CV or test sets.
6. **Threshold optimization** → find threshold maximizing F1 or ROC-based criterion.
7. **Wrap pipeline in ThresholdClassifier** → seamlessly incorporate threshold into predictions.
8. **Evaluate final metrics** → accuracy, F1, ROC-AUC, confusion matrix, classification report.

---

### **Key Conceptual Points**

* **Hyperparameter tuning** → optimizes **model weights and structure**.
* **Threshold tuning** → optimizes **decision boundary** for probabilities → class labels.
* **Threshold tuning is post-training**; retraining isn’t necessary.
* **Pipeline + ThresholdClassifier** → clean, reusable abstraction for production-ready predictions.

---



## Details of the class:
Here’s a more detailed snippet with emphasis on **how the optimal threshold is calculated and used**:

```python
class ThresholdClassifier:
    """
    Wraps a trained pipeline or classifier to apply a custom classification threshold.

    - pipeline: the original sklearn pipeline or model (e.g., RandomForest, AdaBoost)
    - threshold: probability cutoff to classify class 1 (default 0.5)
    
    Why we need it:
    Normally, classifiers use 0.5 as the default threshold. 
    But in imbalanced datasets, the best threshold for maximizing metrics like F1 or balancing
    precision/recall may differ. Here, we calculated it via:
        - Cross-validated F1 (best_threshold = threshold maximizing F1 on CV predictions)
        - ROC analysis (Youden's J: tpr - fpr)
    We then pass this optimal threshold to `self.threshold`.
    """
    def __init__(self, pipeline, threshold=0.5):
        self.pipeline = pipeline
        self.threshold = threshold
    
    def fit(self, X, y):
        """Train the underlying pipeline as usual."""
        self.pipeline.fit(X, y)
        return self
    
    def predict_proba(self, X):
        """Return predicted probabilities from the pipeline."""
        return self.pipeline.predict_proba(X)
    
    def predict(self, X):
        """Convert probabilities to class labels using the custom threshold."""
        proba = self.pipeline.predict_proba(X)[:, 1]  # probability of class 1
        return (proba >= self.threshold).astype(int)

    def score(self, X, y):
        """Compute F1-score for predictions using the custom threshold."""
        from sklearn.metrics import f1_score
        return f1_score(y, self.predict(X))
```

**Summary of flow with thresholds:**

1. Train the pipeline as usual (`.fit()` on training data).
2. Generate predicted probabilities via `cross_val_predict` on training data.
3. Compute **optimal threshold** using either:

   * Maximum F1 across thresholds
   * ROC-based Youden's J statistic (`tpr - fpr`)
4. Pass this threshold to `ThresholdClassifier` via `self.threshold`.
5. Use `.predict()` to get **adjusted class labels**, without retraining the model.

This is **key**: it lets us **modify the decision boundary post-training** to improve F1/recall/precision, which is especially useful for imbalanced datasets like yours.

If you want, I can make a **mini flow diagram/code summary** showing **from tuning → threshold calculation → wrapping in ThresholdClassifier → final predictions** in 10–12 lines. It makes it super easy to reference later. Do you want me to do that?


In [ ]:
# Wrap your best pipeline and set your custom threshold
rf_with_threshold = ThresholdClassifier(best_models["Random Forest"], threshold=0.43)
ada_with_threshold = ThresholdClassifier(best_models["AdaBoost"], threshold=0.48)

# Train (optional if pipeline already trained)
rf_with_threshold.fit(X_train, y_train)

# Predict with custom threshold
y_pred_rf = rf_with_threshold.predict(X_test)
y_pred_ada = ada_with_threshold.predict(X_test)

# Get metrics on these predictions
# ...


In [ ]:
proba_rf = best_models["Random Forest"].predict_proba(X_test)[:, 1]
y_pred_rf = (proba_rf >= 0.43).astype(int)


Rough work:

In [ ]:
rf_params = {
    "model__n_estimators": [400, 600, 800, 1000,],
    "model__max_depth": [None, 10, 30],
    "model__max_features": ["sqrt", "log2", 5, 6, 7,8],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 3, 4, 5],
    "model__bootstrap": [True, False],
    "model__criterion": ["gini", "entropy", "log_loss"]
}
ada_params = {
    "model__n_estimators": [50,70,100,150,200, 300, 400],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0, 1.2],
    "model__algorithm":['SAMME'],
    "model__estimator": [
        DecisionTreeClassifier(max_depth=1),
        DecisionTreeClassifier(max_depth=2),
        DecisionTreeClassifier(max_depth=3),
        DecisionTreeClassifier(max_depth=4),
    ]
}

randomcv_models = [
    ("Random Forest",
     RandomForestClassifier(class_weight="balanced_subsample"),
     rf_params),
    ("AdaBoost",
     AdaBoostClassifier(random_state=42),
     ada_params)
]

best_models = {}

for name, model, params in randomcv_models:
    print(f"\n🔎 Tuning {name}...")

    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    random_search = RandomizedSearchCV(
        pipe,
        param_distributions=params,
        n_iter=75,
        scoring='f1_macro',
        cv=3,
        verbose=1,
        n_jobs=-1,
        random_state=42
    )


    random_search.fit(X_train, y_train)

    print("Best Params:", random_search.best_params_)
    print("Best CV F1:", random_search.best_score_)

    best_models[name] = random_search.best_estimator_


In [ ]:
best_models

In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TRAIN METRICS =====
    y_train_pred = best_pipe.predict(X_train)

    try:
        y_train_proba = best_pipe.predict_proba(X_train)[:, 1]
    except AttributeError:
        y_train_proba = None

    train_metrics = get_classification_metrics(
        y_train, y_train_pred, y_train_proba, verbose=True
    )

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
results = {}

for name, best_pipe in best_models.items():
    print(f"\n🚀 Training {name} with best params...")

    # Train on full training set
    best_pipe.fit(X_train, y_train)

    # ===== TEST METRICS =====
    y_test_pred = best_pipe.predict(X_test)

    try:
        y_test_proba = best_pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        y_test_proba = None

    test_metrics = get_classification_metrics(
        y_test, y_test_pred, y_test_proba, verbose=True
    )

    results[name] = {
        "train": train_metrics,
        "test": test_metrics
    }


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

class ThresholdClassifier:
    def __init__(self, pipeline, threshold=0.5):
        self.pipeline = pipeline
        self.threshold = threshold
    
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)
    
    def predict(self, X):
        proba = self.pipeline.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)
    
    def score(self, X, y):
        y_pred = self.predict(X)
        return f1_score(y, y_pred)


In [ ]:
# Using thresholds you found from CV
rf_threshold = 0.43
ada_threshold = 0.48

rf_with_threshold = ThresholdClassifier(best_models["Random Forest"], threshold=rf_threshold)
ada_with_threshold = ThresholdClassifier(best_models["AdaBoost"], threshold=ada_threshold)


In [ ]:
# Train on full training set (optional if already trained)
rf_with_threshold.fit(X_train, y_train)
ada_with_threshold.fit(X_train, y_train)

# Predict on test set with custom thresholds
y_pred_rf = rf_with_threshold.predict(X_test)
y_pred_ada = ada_with_threshold.predict(X_test)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

def print_metrics(y_true, y_pred, y_proba=None):
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")
    if y_proba is not None:
        print(f"ROC AUC  : {roc_auc_score(y_true, y_proba):.4f}")

print("=== Random Forest with threshold ===")
print_metrics(y_test, y_pred_rf, rf_with_threshold.predict_proba(X_test)[:, 1])

print("\n=== AdaBoost with threshold ===")
print_metrics(y_test, y_pred_ada, ada_with_threshold.predict_proba(X_test)[:, 1])
